# Atividade 02 — Regras de Associação (Apriori)

**Tema:** descobrir padrões em pedidos de uma **loja online de moda**.

O algoritmo Apriori encontra itens que aparecem juntos com frequência e gera regras do tipo *se A então B*.

> Compatível com Google Colab — basta executar todas as células.

In [ ]:
!pip install -q mlxtend

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
import logging, warnings
logging.getLogger().setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

In [ ]:
dados = pd.DataFrame({
    "Cliente": [f"CL{i:02d}" for i in range(1, 13)],
    "Canal": ["App", "App", "Site", "Loja", "Loja", "App",
              "Site", "App", "Loja", "Site", "App", "Loja"],
    "Pagamento": ["Pix", "Cartao", "Cartao", "Dinheiro", "Dinheiro", "Pix",
                  "Boleto", "Pix", "Dinheiro", "Boleto", "Cartao", "Pix"],
    "Frete": ["Expresso", "Expresso", "Normal", "Retirada", "Retirada", "Expresso",
              "Normal", "Expresso", "Retirada", "Normal", "Expresso", "Retirada"],
    "Cupom": ["Sim", "Sim", "Nao", "Nao", "Nao", "Sim",
              "Nao", "Sim", "Nao", "Nao", "Sim", "Nao"],
    "Resultado": ["Fidelizado", "Fidelizado", "Fidelizado", "Abandonou", "Abandonou", "Fidelizado",
                  "Abandonou", "Fidelizado", "Abandonou", "Abandonou", "Fidelizado", "Fidelizado"]
})
dados

In [ ]:
transacoes = []
for _, linha in dados.iterrows():
    transacoes.append([
        f"Canal={linha['Canal']}",
        f"Pagamento={linha['Pagamento']}",
        f"Frete={linha['Frete']}",
        f"Cupom={linha['Cupom']}",
        f"Resultado={linha['Resultado']}"
    ])
transacoes

In [ ]:
te = TransactionEncoder()
matriz = te.fit(transacoes).transform(transacoes)
dados_transformados = pd.DataFrame(matriz, columns=te.columns_)
dados_transformados

In [ ]:
itemsets = apriori(dados_transformados, min_support=0.25, use_colnames=True)
itemsets.sort_values(by="support", ascending=False)

In [ ]:
regras = association_rules(itemsets, metric="confidence", min_threshold=0.65)
resumo = regras[["antecedents", "consequents", "support", "confidence", "lift"]]
resumo.sort_values(by="confidence", ascending=False)

In [ ]:
def formatar_conjunto(conjunto):
    return ", ".join(sorted(list(conjunto)))

resumo = resumo.copy()
resumo["Regra"] = resumo.apply(
    lambda r: f"{formatar_conjunto(r['antecedents'])} -> {formatar_conjunto(r['consequents'])}",
    axis=1
)
resumo[["Regra", "support", "confidence", "lift"]].sort_values(by="confidence", ascending=False)

In [ ]:
fidelizados = resumo[resumo["consequents"].apply(lambda x: "Resultado=Fidelizado" in x)]
fidelizados[["Regra", "support", "confidence", "lift"]]

In [ ]:
abandonos = resumo[resumo["consequents"].apply(lambda x: "Resultado=Abandonou" in x)]
abandonos[["Regra", "support", "confidence", "lift"]]

## Conclusão

Regras com **lift > 1** indicam associação positiva entre antecedente e consequente. A loja pode usar isso para montar combos (Pix + Frete Expresso + Cupom) que aumentem a fidelização.